# End-to-End Tiered Data Management Pipeline (Colab Runner)

Reproducing a lightweight, modular prototype of **Tiered Data Management** (arXiv:2602.09003).
- Repository: `https://github.com/mahendravelagapudi099-wq/Ultra-Data.git`

### Pipeline Overview:
- **Phase 1 (L1):** Heuristic text cleaning, length/ratio filtering & exact deduplication.
- **Phase 2 (L2):** Weak demo labeling, TF-IDF + numeric feature selector, scoring & selection.
- **Phase 3 (L3):** Deterministic offline LLM refinement, Q&A synthesis & textbook chapters.
- **Phase 3.5 (L4):** Knowledge unit validation & structured export with provenance metadata.
- **Phase 4:** Cross-tier evaluation, retention analysis & report generation.

In [1]:
# Cell 1: Clone repository (recommended) or mount Google Drive
import os

repo_url = "https://github.com/mahendravelagapudi099-wq/Ultra-Data.git"
target_dir = "/content/Ultra-Data"

if not os.path.exists(target_dir):
    !git clone {repo_url} {target_dir}
else:
    print(f"{target_dir} already exists. Pulling latest updates...")
    !cd {target_dir} && git pull origin main

# Optional: Mount Google Drive if syncing via Drive
# from google.colab import drive
# drive.mount('/content/drive')

Cloning into '/content/Ultra-Data'...
remote: Enumerating objects: 54, done.
remote: Counting objects: 100% (54/54), done.
remote: Compressing objects: 100% (44/44), done.
remote: Total 54 (delta 7), reused 54 (delta 7), pack-reused 0 (from 0)
Receiving objects: 100% (54/54), 809.10 KiB | 18.82 MiB/s, done.
Resolving deltas: 100% (7/7), done.


In [2]:
# Cell 2: Change directory into the project root
import os

candidate_paths = [
    "/content/Ultra-Data",
    "/content/drive/MyDrive/Ultra-Data",
    "/content/drive/MyDrive/Ultra-Dataa",
]

for path in candidate_paths:
    if os.path.exists(path):
        os.chdir(path)
        print(f"Active project root: {path}")
        break
else:
    print(f"Current working directory: {os.getcwd()}")

!pwd
!ls -la

Active project root: /content/Ultra-Data
/content/Ultra-Data
total 76
drwxr-xr-x 10 root root 4096 Sep 14 10:05 .
drwxr-xr-x  1 root root 4096 Sep 14 10:05 ..
-rw-r--r--  1 root root 3802 Sep 14 10:05 AGENTS.md
drwxr-xr-x  2 root root 4096 Sep 14 10:05 colab
drwxr-xr-x  2 root root 4096 Sep 14 10:05 configs
drwxr-xr-x  8 root root 4096 Sep 14 10:05 data
drwxr-xr-x  8 root root 4096 Sep 14 10:05 .git
-rw-r--r--  1 root root  432 Sep 14 10:05 .gitignore
drwxr-xr-x  2 root root 4096 Sep 14 10:05 paper
-rw-r--r--  1 root root 4484 Sep 14 10:05 PHASES.md
-rw-r--r--  1 root root 4480 Sep 14 10:05 README.md
drwxr-xr-x  2 root root 4096 Sep 14 10:05 reports
-rw-r--r--  1 root root 2690 Sep 14 10:05 requirements.lock.txt
-rw-r--r--  1 root root  212 Sep 14 10:05 requirements.txt
-rw-r--r--  1 root root 2915 Sep 14 10:05 RESULTS_TEMPLATE.md
drwxr-xr-x  2 root root 4096 Sep 14 10:05 scripts
drwxr-xr-x  5 root root 4096 Sep 14 10:05 src


In [3]:
# Cell 3: Install required packages if missing
!pip install -q -r requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 151.9/151.9 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 837.9/837.9 kB 17.4 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 55.7 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.4/322.4 kB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 947.5/947.5 kB 22.6 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.5/77.5 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.8/59.8 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.7/296.7 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 43.0 MB/s eta 0:00:0000:0100:01


In [4]:
# Cell 4: Set PYTHONPATH
import sys
import os

project_root = os.getcwd()
if project_root not in sys.path:
    sys.path.insert(0, project_root)

os.environ["PYTHONPATH"] = "."
print(f"Project root (CWD): {project_root}")
print(f"PYTHONPATH: {os.environ.get('PYTHONPATH')}")

Project root (CWD): /content/Ultra-Data
PYTHONPATH: .


In [ ]:
# Fetch real web sample from openbmb/Ultra-FineWeb (300 records)
# Note: If this cell fails or times out due to Hugging Face network/rate limits,
# the next cell will automatically fall back to local substitute mock data.
!PYTHONPATH=. python scripts/load_real_data.py


In [5]:
# Cell 5: Run Phase 1 (L0 Generation + L1 Heuristic Filtering)
!PYTHONPATH=. python scripts/generate_l0_expanded.py
!PYTHONPATH=. python scripts/run_l1.py --config configs/l1_expanded.yaml

Generated 145 raw substitute records at data/l0_raw/l0_expanded_SUBSTITUTE.jsonl
Source: local_substitute_l1
───────────────────── L1 Tiny Demo — Tier 1 Data Cleaning ──────────────────────
Config: configs/l1_expanded.yaml
L1 processing: 100% 145/145 [00:00<00:00, 10142.66doc/s]
         L1 Pipeline Statistics          
┏━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┓
┃ Metric             ┃            Value ┃
┡━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━┩
│ Input Count        │              145 │
│ Output Count       │               33 │
│ Removed (filtered) │              102 │
│ Duplicates Removed │               10 │
│ Source Used        │ local_substitute │
└────────────────────┴──────────────────┘
     Filter Reasons      
┏━━━━━━━━━━━━━━━┳━━━━━━━┓
┃ Reason        ┃ Count ┃
┡━━━━━━━━━━━━━━━╇━━━━━━━┩
│ duplicate     │    10 │
│ empty         │     9 │
│ pass          │    43 │
│ too_few_words │    10 │
│ too_short     │    83 │
└───────────────┴───────┘

Output Files:
  Parquet: data/l1_filter

In [6]:
# Cell 6: Run Phase 2 (L2 Model-Driven Selection)
!PYTHONPATH=. python scripts/run_l2.py --config configs/l2_tiny.yaml

─────────────────── L2 Demo — Tier 2 Model-Driven Selection ────────────────────
Config: configs/l2_tiny.yaml
         L2 Selection Statistics          
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┓
┃ Metric                        ┃  Value ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━┩
│ Input Records (L1)            │     33 │
│ Selected Records (L2)         │     16 │
│ Selection Rate                │  48.5% │
│ Selector Train Accuracy       │  95.7% │
│ Selector Test Accuracy        │ 100.0% │
│ Mean Quality Score (All)      │ 0.3570 │
│ Mean Quality Score (Selected) │ 0.6605 │
└───────────────────────────────┴────────┘

Output Files:
  All Scored (Parquet): data/l2_scores/l2_selected_all_scored.parquet
  Selected (Parquet):   data/l2_selected/l2_selected.parquet
  Selected (JSONL):     data/l2_selected/l2_selected.jsonl
───────────────────────────────────── Done ─────────────────────────────────────


In [7]:
# Cell 7: Run Phase 3 (L3 LLM Refinement & Synthesis) & Phase 3.5 (L4 Knowledge Export)
!PYTHONPATH=. python scripts/run_l3.py --config configs/l3_tiny.yaml
!PYTHONPATH=. python scripts/run_l4_export.py --config configs/l4_tiny.yaml

───────────────── L3 Demo — Tier 3 LLM Refinement & Synthesis ──────────────────
Config: configs/l3_tiny.yaml
L3 Refinement: 100% 16/16 [00:00<00:00, 3052.62doc/s]
                        L3 Refinement Statistics                         
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Metric                      ┃                                   Value ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ Input Records (L2 Selected) │                                      16 │
│ Refined Records (L3)        │                                      16 │
│ Synthesis Generator         │ MockLLMProvider (deterministic offline) │
└─────────────────────────────┴─────────────────────────────────────────┘

Output Files:
  Parquet: data/l3_refined/l3_refined.parquet
  JSONL:   data/l3_refined/l3_refined.jsonl
───────────────────────────────────── Done ─────────────────────────────────────
───────────────── L4 Demo — Tier 4 Organized Knowledge Exp

In [8]:
# Cell 8: Run Phase 4 Evaluation and Comparison
!PYTHONPATH=. python scripts/run_evaluation.py

───────────────── Pipeline Evaluation — Cross-Tier Progression ─────────────────
                         Cross-Tier Progression Summary                         
┏━━━━━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━┓
┃             ┃           ┃           ┃             ┃      Symbol ┃            ┃
┃ Tier        ┃ Doc Count ┃ Avg Words ┃ Alpha Ratio ┃       Ratio ┃ Duplicates ┃
┡━━━━━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━┩
│ L0_Raw      │       145 │      24.0 │      0.7996 │      0.0762 │         10 │
│ L1_Filtered │        33 │      38.2 │      0.8428 │      0.0248 │          0 │
│ L2_Selected │        16 │      43.8 │      0.8377 │      0.0251 │          0 │
│ L3_Refined  │        16 │      43.8 │      0.8377 │      0.0251 │          0 │
│ L4_Organiz… │        16 │      49.4 │      0.8251 │      0.0349 │          0 │
└─────────────┴───────────┴───────────┴─────────────┴─────────────┴────────────┘

Generated Reports:
  Markdo

In [9]:
# Cell 9: Print report locations and display summary
from pathlib import Path

report_path = Path("reports/pipeline_summary.md")
if report_path.exists():
    print(report_path.read_text(encoding="utf-8"))
else:
    print("Report not found at reports/pipeline_summary.md. Ensure Phase 4 ran successfully.")

# Pipeline Evaluation & Tier Comparison Report

> [!NOTE]
> **Notice:** This report reflects a small-scale, offline demo reproduction of the methodology from
> *"Data Science and Technology Towards AGI Part I: Tiered Data Management"* (arXiv:2602.09003).
> Heuristics, thresholds, and mock LLM synthesizers are lightweight starter implementations and do not claim paper-scale results.

**Generated:** 2026-09-14 10:06:18 UTC  
**Environment:** Colab / Linux / Local agnostic  

## 1. Tier-by-Tier Quality Progression

| Tier | Description | Doc Count | Avg Chars | Avg Words | Alpha Ratio | Symbol Ratio | Duplicates |
|---|---|---|---|---|---|---|---|
| `L0_Raw` | Unfiltered substitute web corpus | 145 | 181.9 | 24.0 | 0.7996 | 0.0762 | 10 |
| `L1_Filtered` | Heuristic filtered & exact deduped | 33 | 282.8 | 38.2 | 0.8428 | 0.0248 | 0 |
| `L2_Selected` | Model-selected informative tokens | 16 | 317.8 | 43.8 | 0.8377 | 0.0251 | 0 |
| `L3_Refined` | Mock LLM refined, Q&A & textbook | 16 | 317.8